# 06 · Evaluation, experiments, error analysis and LM ablation

**Inputs:** `splits.csv` and the `results/<run>/` folders from notebooks 03–05 (attach each run's
notebook output). Runs that are missing are simply skipped.

Rules enforced here:
* The **final model is selected on validation WER** among eligible runs; the test set is never used to choose.
* Models are compared on the **identical** test utterances (paired speaker bootstrap); the random-split run
  is reported separately because its test set is different by construction.
* Confidence intervals resample **speakers**, not utterances.

CPU is enough unless you re-decode many runs with the LM.

In [ ]:
# ── Setup: locate the project source (src/dholuo_asr) and the output folder ──────────
import os, sys, subprocess, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# After pushing the repo to GitHub, put its URL here (Kaggle: Settings → Internet ON).
# Alternatively upload the repo as a private Kaggle dataset and attach it as an input.
REPO_URL = os.environ.get("DHOLUO_REPO_URL", "https://github.com/YOUR-USERNAME/dholuo-asr.git")

def _find_src(max_depth=4):
    for root in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working"), Path("/kaggle/input")]:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth or len(filenames) > 500:
                dirnames[:] = []          # don't descend into audio folders
            if Path(dirpath).name == "dholuo_asr" and "__init__.py" in filenames:
                return Path(dirpath).parent
    return None

SRC = _find_src()
if SRC is None and "YOUR-USERNAME" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/dholuo-asr"], check=True)
    SRC = Path("/kaggle/working/dholuo-asr/src")
if SRC is None:
    raise RuntimeError("Project source not found: set REPO_URL (Internet ON) or attach the repo as a dataset.")
sys.path.insert(0, str(SRC))
REPO = SRC.parent
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path(os.environ.get("DHOLUO_OUT", "/kaggle/working" if ON_KAGGLE else REPO / "outputs"))
OUT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOTS = [OUT, Path("/kaggle/input")]
print("source :", SRC)
print("outputs:", OUT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
DIALECT_COLORS = {"Milambo": "#2a6f97", "Nyanduat": "#e07a5f"}
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", bbox_inches="tight"); plt.show()

In [ ]:
from collections import Counter
from dholuo_asr.utils import find_artifact, dump_json
from dholuo_asr.metrics import score_frame, corpus_rates, group_rates, speaker_bootstrap, paired_speaker_bootstrap
splits = pd.read_csv(find_artifact("splits.csv", ARTIFACT_ROOTS), keep_default_na=False, na_values=[""])
meta_cols = ["utt_id", "speaker", "dialect", "gender", "age", "domain", "duration_s", "prompt_in_train", "cs_candidate", "n_words", "flag_fast_speech"]
EVAL = OUT / "evaluation"; EVAL.mkdir(exist_ok=True)

runs = {}
for root in ARTIFACT_ROOTS:
    if not Path(root).exists():
        continue
    for mf in Path(root).rglob("metrics.json"):
        if mf.parent.parent.name != "results":
            continue
        name = mf.parent.name
        if name not in runs or mf.stat().st_mtime > (runs[name] / "metrics.json").stat().st_mtime:
            runs[name] = mf.parent
print(f"{len(runs)} runs found:", sorted(runs))

def load_preds(name, split):
    p = pd.read_csv(runs[name] / f"predictions_{split}.csv", keep_default_na=False, na_values=[""])
    p = p[["utt_id", "ref", "hyp"]].merge(splits[[c for c in meta_cols if c in splits]], on="utt_id", how="left")
    p["hyp"] = p["hyp"].fillna(""); p["ref"] = p["ref"].fillna("")
    return score_frame(p)

scored = {(n, s): load_preds(n, s) for n in runs for s in ("val", "test") if (runs[n] / f"predictions_{s}.csv").exists()}
cfgs = {n: json.loads((runs[n] / "config.json").read_text()) if (runs[n] / "config.json").exists() else {} for n in runs}
mets = {n: json.loads((runs[n] / "metrics.json").read_text()) for n in runs}

## 1. Master results table (all runs)

In [ ]:
rows = []
for n in sorted(runs):
    r = {"run": n}
    c = cfgs.get(n, {})
    r["family/strategy"] = f"{c.get('model', {}).get('checkpoint', n)} / {c.get('model', {}).get('strategy', '-')}"
    r["split"] = c.get("data", {}).get("split_scheme", "speaker")
    for s in ("val", "test"):
        if (n, s) in scored:
            cr = corpus_rates(scored[(n, s)]); ci = speaker_bootstrap(scored[(n, s)])
            r[f"{s}_WER"] = cr["wer"]; r[f"{s}_CER"] = cr["cer"]
            if s == "test":
                r["test_WER_95CI"] = f"[{ci['wer_ci'][0]:.3f}, {ci['wer_ci'][1]:.3f}]"
                sub = scored[(n, s)]; unseen = sub[sub["prompt_in_train"].astype(str).str.lower() == "false"]
                r["test_WER_unseen_prompts"] = corpus_rates(unseen)["wer"] if len(unseen) else np.nan
    m = mets[n]
    r["trainable_params_M"] = m.get("params_trainable", np.nan) / 1e6 if m.get("params_trainable") else np.nan
    r["train_min"] = m.get("best", {}).get("train_minutes", np.nan)
    r["best_step"] = m.get("best", {}).get("step", np.nan)
    rows.append(r)
master = pd.DataFrame(rows).set_index("run")
master.to_csv(EVAL / "master_results.csv")
master.round(4)

## 2. Final model selection — validation set only

In [ ]:
# Candidate final models (ablation runs such as data-size, random-split and seed repeats are excluded by design)
ELIGIBLE = os.environ.get("ELIGIBLE", "xlsr_full,mms_adapter,xlsr_lora,xlsr_frozen,bilstm_ctc,xlsr_full_minimal_text,mms_zeroshot").split(",")
cand = master.loc[[n for n in ELIGIBLE if n in master.index and not np.isnan(master.loc[n, "val_WER"])]]
PRIMARY = cand["val_WER"].idxmin()
print("selected on validation WER:", PRIMARY)
print(cand[["val_WER", "val_CER"]].sort_values("val_WER").round(4))

## 3. Experiment C & G — model family and fine-tuning strategy
Paired speaker bootstrap of each model against the selected model on the same test utterances. A CI excluding 0 means the difference is robust to which speakers happened to be in the test set.

In [ ]:
def paired_table(names, ref=PRIMARY, split="test"):
    out = []
    for n in names:
        if (n, split) not in scored or n == ref:
            continue
        try:
            p = paired_speaker_bootstrap(scored[(n, split)], scored[(ref, split)])
            out.append({"run": n, "WER": corpus_rates(scored[(n, split)])["wer"], f"WER - {ref}": p["wer_diff"],
                        "95% CI": f"[{p['ci'][0]:+.3f}, {p['ci'][1]:+.3f}]"})
        except ValueError as e:
            out.append({"run": n, "note": str(e)})
    return pd.DataFrame(out)
expC = paired_table(["mms_zeroshot", "bilstm_ctc", "xlsr_full", "mms_adapter", "xlsr_frozen", "xlsr_lora"])
display(expC)
g = master.loc[[n for n in ["xlsr_frozen", "xlsr_lora", "xlsr_full", "mms_adapter"] if n in master.index]]
if len(g):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].scatter(g["trainable_params_M"], g["test_WER"]); ax[1].scatter(g["train_min"], g["test_WER"])
    for n, r in g.iterrows():
        ax[0].annotate(n, (r["trainable_params_M"], r["test_WER"])); ax[1].annotate(n, (r["train_min"], r["test_WER"]))
    ax[0].set_xscale("log"); ax[0].set_xlabel("trainable parameters (M, log)"); ax[1].set_xlabel("training minutes")
    for a in ax: a.set_ylabel("test WER")
    savefig("expG_tradeoff")

## 4. Experiment A — training-data size (speakers)

In [ ]:
A = [("xlsr_full_frac10", .10), ("xlsr_full_frac25", .25), ("xlsr_full_frac50", .50), ("xlsr_full", 1.0)]
pts = []
for n, f in A:
    if (n, "test") in scored:
        s = scored[(n, "test")]; ci = speaker_bootstrap(s)
        pts.append({"run": n, "frac_speakers": f, "train_hours": mets[n]["data"]["train"]["hours"],
                    "train_speakers": mets[n]["data"]["train"]["speakers"], "WER": corpus_rates(s)["wer"],
                    "lo": ci["wer_ci"][0], "hi": ci["wer_ci"][1], "CER": corpus_rates(s)["cer"]})
pts = pd.DataFrame(pts)
if len(pts):
    display(pts.round(4))
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.errorbar(pts["train_hours"], pts["WER"], yerr=[pts["WER"] - pts["lo"], pts["hi"] - pts["WER"]], marker="o", capsize=3, label="WER")
    ax.plot(pts["train_hours"], pts["CER"], marker="s", label="CER")
    ax.set_xscale("log"); ax.set_xlabel("training hours (log)"); ax.set_ylabel("test error rate"); ax.legend()
    savefig("expA_data_size")

## 5. Experiment B — transcript normalisation
Both runs are scored with the same evaluation normaliser, so the comparison isolates the *training* transcript view.

In [ ]:
if ("xlsr_full_minimal_text", "test") in scored and ("xlsr_full", "test") in scored:
    p = paired_speaker_bootstrap(scored[("xlsr_full_minimal_text", "test")], scored[("xlsr_full", "test")])
    print("WER(minimal) - WER(normalised) =", round(p["wer_diff"], 4), "95% CI", tuple(round(x, 4) for x in p["ci"]))
    # orthographic (case/punctuation-sensitive) error of the minimal-view model, for completeness
    from dholuo_asr.metrics import word_align
    s = scored[("xlsr_full_minimal_text", "test")]
    ortho = sum(word_align(r, h, normalise=False).errors for r, h in zip(s["ref"], s["hyp"])) / max(sum(len(r.split()) for r in s["ref"]), 1)
    print("orthographic WER of the minimal-view model (no eval normalisation):", round(ortho, 4))
else:
    print("run xlsr_full_minimal_text and xlsr_full to report Experiment B")

## 6. Experiment F — what speaker leakage does to the numbers

In [ ]:
if ("xlsr_full_random_split", "test") in scored and ("xlsr_full", "test") in scored:
    a, b = scored[("xlsr_full_random_split", "test")], scored[("xlsr_full", "test")]
    f = pd.DataFrame([{"evaluation": "random row split (speakers shared)", **corpus_rates(a), **speaker_bootstrap(a)},
                      {"evaluation": "speaker-independent (official)", **corpus_rates(b), **speaker_bootstrap(b)}]).set_index("evaluation")
    display(f[["wer", "cer", "wer_ci", "n_utts"]])
    print(f"relative WER optimism of the leaky split: {(b['w_err'].sum()/b['w_ref'].sum()) / max(a['w_err'].sum()/a['w_ref'].sum(), 1e-9):.2f}x")
else:
    print("run xlsr_full_random_split to report Experiment F")

## 7. Experiment D/E and subgroups — selected model
Dialect (D), code-switching candidates (E), gender, age, utterance length and prompt novelty.

In [ ]:
from dholuo_asr.errors import duration_buckets
S = scored[(PRIMARY, "test")].copy()
S["duration_bucket"] = duration_buckets(S)
S["prompt_seen_in_train"] = S["prompt_in_train"].astype(str).str.lower().map({"true": "seen", "false": "unseen"})
S["code_switch_candidate"] = S["cs_candidate"].astype(str).str.lower().map({"true": "yes", "false": "no"})
sub_tables = {}
for col in ["dialect", "gender", "age", "domain", "duration_bucket", "prompt_seen_in_train", "code_switch_candidate"]:
    if col in S and S[col].notna().any():
        t = group_rates(S, col)[["wer", "cer", "n_utts", "n_speakers", "n_ref_words"]]
        if col == "dialect":   # CIs per dialect, resampling that dialect's speakers
            t["wer_95ci"] = [tuple(round(x, 3) for x in speaker_bootstrap(S[S[col] == k])["wer_ci"]) for k in t.index]
        sub_tables[col] = t
        print(f"\n=== {col} ==="); display(t.round(4))
print("NOTE: code-switch flags are", "VERIFIED" if json.loads(find_artifact("split_report.json", ARTIFACT_ROOTS).read_text())["cs_verified"] else "UNVERIFIED heuristic candidates")
spk = group_rates(S, "speaker").join(S.groupby("speaker")["dialect"].first()).sort_values("wer")
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(len(spk)), spk["wer"], color=[DIALECT_COLORS.get(d, "grey") for d in spk["dialect"]])
ax.set_xticks(range(len(spk))); ax.set_xticklabels(spk.index, rotation=90, fontsize=7); ax.set_ylabel("WER"); ax.set_title(f"Per-speaker test WER — {PRIMARY}")
savefig("per_speaker_wer")

## 8. Error analysis — what fails, how often, where, and why

In [ ]:
from dholuo_asr.errors import ref_word_outcomes, segmentation_errors, char_confusions, word_substitutions, qualitative_sample
train_text = splits.loc[splits.split == "train", "text_norm"]
train_counts = Counter(w for t in train_text for w in str(t).split())
W = ref_word_outcomes(S, train_counts)
tot = S[["w_sub", "w_ins", "w_del"]].sum(); print("error composition:", (tot / tot.sum()).round(3).to_dict())
fb = W.groupby("freq_bucket").agg(ref_words=("word", "size"), error_rate=("correct", lambda c: 1 - c.mean()))
fb = fb.reindex(["unseen (OOV)", "1-2", "3-10", "11-100", ">100"]).dropna(how="all")
display(fb.round(3))
ap = W.groupby("has_apostrophe")["correct"].agg(["size", lambda c: 1 - c.mean()]).rename(columns={"size": "ref_words", "<lambda_0>": "error_rate"})
print("words containing an apostrophe (e.g. ng'):"); display(ap.round(3))
seg = segmentation_errors(S)
print(f"word-merge errors: {seg['merges'].sum()}, word-split errors: {seg['splits'].sum()}, utterances wrong ONLY by spacing: {int(seg['spacing_only'].sum())}")
print("top character confusions:"); display(char_confusions(S))
print("top word substitutions:"); display(word_substitutions(W))
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(fb.index.astype(str), fb["error_rate"]); ax.set_ylabel("word error rate"); ax.set_xlabel("frequency of the word in training transcripts")
savefig("error_by_train_frequency")

In [ ]:
# Error categories (non-exclusive, utterance level), quantified
cats = pd.DataFrame({"utt_id": S["utt_id"], "has_error": S["w_err"] > 0})
oov_utts = set(W.loc[W["freq_bucket"] == "unseen (OOV)", "utt_id"])
cats["contains_unseen_word"] = S["utt_id"].isin(oov_utts)
cats["short_utt(<2s)"] = S["duration_s"] < 2
cats["long_utt(>10s)"] = S["duration_s"] > 10
cats["spacing_only"] = seg["spacing_only"].to_numpy()
cats["unseen_prompt"] = S["prompt_seen_in_train"] == "unseen"
cats["code_switch_candidate"] = S["code_switch_candidate"] == "yes"
if "flag_fast_speech" in S: cats["fast_speech_flag"] = S["flag_fast_speech"].astype(str).str.lower() == "true"
cat_tab = pd.DataFrame({c: {"utterances": int(cats[c].sum()), "of which with errors": int((cats[c] & cats.has_error).sum()),
                            "WER within category": corpus_rates(S[cats[c].to_numpy()])["wer"] if cats[c].any() else np.nan}
                        for c in cats.columns if c not in ("utt_id", "has_error")}).T
display(cat_tab.round(3)); cat_tab.to_csv(EVAL / "error_categories.csv")

### Qualitative examples (deterministic, not cherry-picked)
Best / median / worst / random utterances per dialect. `qualitative_examples.csv` has an empty `annotator_notes` column: as a Dholuo speaker, annotate *why* each error happened (dialectal form, spelling variant, mis-hearing, transcript error…) — that linguistic judgement is what the report's qualitative section needs.

In [ ]:
Q = qualitative_sample(S, by="dialect", per_group=3)
Q[["sample_reason", "dialect", "speaker", "duration_s", "utt_wer", "ref", "hyp", "annotator_notes"]].to_csv(EVAL / "qualitative_examples.csv", index=False)
with pd.option_context("display.max_colwidth", 120):
    display(Q[["sample_reason", "dialect", "utt_wer", "ref", "hyp"]])

## 9. Ablation — n-gram language model in decoding
**Single factor isolated:** the acoustic model's log-probabilities are fixed (saved by notebook 05); only the decoder changes:
greedy → beam search without LM → beam search + word trigram LM (trained on **training transcripts only**, `src/dholuo_asr/lm.py`).
LM weight α and word bonus β are tuned on **validation**. Because read prompts can recur across speakers, the
LM can memorise test sentences that also appear in training — so the **unseen-prompt** column is the honest one.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyctcdecode", "pygtrie"], check=False)
try:
    import kenlm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "https://github.com/kpu/kenlm/archive/master.zip"], check=False)
from pyctcdecode import build_ctcdecoder
try:
    import kenlm  # noqa: F401
    HAVE_KENLM = True
except ImportError:
    HAVE_KENLM = False
print("kenlm available:", HAVE_KENLM)

In [ ]:
from dholuo_asr.lm import NGramLM
from dholuo_asr.text import apply_view, eval_normalise
LM_RUN = os.environ.get("LM_RUN", PRIMARY)
rd = runs[LM_RUN]
ok = (rd / "logprobs_val.npz").exists() and (rd / "logprobs_test.npz").exists()
if not ok:
    print(f"{LM_RUN} has no saved log-probabilities (output.keep_logprobs=false or zero-shot run) — ablation skipped")
else:
    view = cfgs[LM_RUN]["data"]["text_view"]
    vocab = json.loads((rd / "vocab.json").read_text())
    labels = [""] * len(vocab)
    for ch, i in vocab.items():
        labels[i] = {"[PAD]": "", "|": " ", "[UNK]": "⁇"}.get(ch, ch)
    lm_train = [apply_view(t, view).split() for t in splits.loc[splits.split == "train", "text_raw"]]
    lm = NGramLM(order=3).fit(lm_train)
    arpa = lm.write_arpa(EVAL / "train_trigram.arpa")
    val_txt = [apply_view(t, view).split() for t in splits.loc[splits.split == "val", "text_raw"]]
    print(f"trigram LM: {len(lm.vocab):,} words | val perplexity {lm.perplexity(val_txt):.1f}")
    LP = {s: np.load(rd / f"logprobs_{s}.npz") for s in ("val", "test")}

    def decode_all(decoder, split, beam=32):
        ref = scored[(LM_RUN, split)]
        out = ref[["utt_id", "ref", "speaker", "dialect", "prompt_in_train"]].copy()
        out["hyp"] = [" ".join(decoder.decode(LP[split][u].astype(np.float32), beam_width=beam).split()) for u in out["utt_id"]]
        return score_frame(out)

    abl = {"greedy": {s: scored[(LM_RUN, s)] for s in ("val", "test")}}
    beam_dec = build_ctcdecoder(labels)
    abl["beam (no LM)"] = {s: decode_all(beam_dec, s) for s in ("val", "test")}
    if HAVE_KENLM:
        lm_dec = build_ctcdecoder(labels, kenlm_model_path=str(arpa), unigrams=lm.unigrams(), alpha=0.5, beta=1.0)
        grid = []
        for a in [0.2, 0.5, 0.8, 1.2]:
            for b in [0.0, 1.0, 2.0]:
                lm_dec.reset_params(alpha=a, beta=b)
                grid.append({"alpha": a, "beta": b, "val_wer": corpus_rates(decode_all(lm_dec, "val", beam=16))["wer"]})
        grid = pd.DataFrame(grid); best_ab = grid.loc[grid.val_wer.idxmin()]
        print("alpha/beta chosen on validation:", best_ab.to_dict())
        lm_dec.reset_params(alpha=float(best_ab.alpha), beta=float(best_ab.beta))
        abl["beam + trigram LM"] = {s: decode_all(lm_dec, s) for s in ("val", "test")}
    rows = []
    for k, d in abl.items():
        t = d["test"]; unseen = t[t["prompt_in_train"].astype(str).str.lower() == "false"]
        rows.append({"decoder": k, "val_WER": corpus_rates(d["val"])["wer"], "test_WER": corpus_rates(t)["wer"],
                     "test_CER": corpus_rates(t)["cer"], "test_WER_unseen_prompts": corpus_rates(unseen)["wer"] if len(unseen) else np.nan,
                     "test_WER_95CI": speaker_bootstrap(t)["wer_ci"]})
    lm_table = pd.DataFrame(rows).set_index("decoder"); display(lm_table.round(4))
    lm_table.to_csv(EVAL / "lm_ablation.csv")

## 10. Seed variation and final summary

In [ ]:
seeds = [n for n in ["xlsr_full", "xlsr_full_seed2", "xlsr_full_seed3"] if (n, "test") in scored]
if len(seeds) > 1:
    w = [corpus_rates(scored[(n, "test")])["wer"] for n in seeds]
    print(f"primary-recipe test WER over {len(seeds)} seeds: mean {np.mean(w):.4f}, std {np.std(w, ddof=1):.4f}, values {np.round(w, 4)}")
final = {"selected_model": PRIMARY, "selection_criterion": "lowest validation WER among eligible runs",
         "test": {**corpus_rates(S), **speaker_bootstrap(S)},
         "subgroups": {k: v.reset_index().to_dict("records") for k, v in sub_tables.items()},
         "master": master.reset_index().to_dict("records")}
dump_json(final, EVAL / "final_summary.json")
print(json.dumps(final["test"], indent=1, default=str))
print("all evaluation outputs in", EVAL)